# 08 — Space syntax dan aksesibilitas jaringan

**Betweenness centrality** mengukur seberapa sering sebuah ruas dilewati jalur terpendek antar-pasangan titik —
prediktor kuat di mana pergerakan terkonsentrasi. Lalu **aksesibilitas jaringan**: jarak ke puskesmas di
sepanjang jalan, bukan garis lurus.

In [ ]:
import sys
sys.path.insert(0, "../scripts")

import time

import geopandas as gpd
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd

from kota import CFG, OSM, PROC, RAW, UTM

G = ox.project_graph(ox.load_graphml(RAW / "jalan_drive.graphml"), to_crs=UTM)
U = ox.convert.to_undirected(G)
print(f"{U.number_of_nodes():,} simpul, {U.number_of_edges():,} ruas (tak berarah)")

## 1. Betweenness dengan sampel
Betweenness eksak menghitung jalur terpendek dari *setiap* simpul — pada 51 ribu simpul itu berjam-jam.
Kita pakai sampel `k` simpul sumber dengan `seed` tetap, supaya hasilnya bisa diulang.

In [ ]:
K, SEED = 1000, 42
t0 = time.time()
eb = nx.edge_betweenness_centrality(U, k=K, weight="length", seed=SEED, normalized=True)
print(f"betweenness {K} sampel: {time.time() - t0:.0f} detik")

edges = ox.graph_to_gdfs(U, nodes=False).reset_index()
edges["betweenness"] = [eb.get((u, v, k), eb.get((v, u, k), 0)) for u, v, k in zip(edges.u, edges.v, edges.key)]
edges["nama"] = edges["name"].apply(lambda n: n if isinstance(n, str) else (", ".join(n) if isinstance(n, list) else None))

In [ ]:
top = edges.groupby("nama")["betweenness"].max().sort_values(ascending=False).head(10)
top.round(4)

Jalan dengan betweenness tertinggi biasanya jalan komersial utama: jaringan membentuk guna lahan.

## 2. Seberapa stabil hasil sampel?
Ulangi dengan seed lain; bila urutan jalan teratas berubah banyak, sampelnya terlalu kecil.

In [ ]:
eb2 = nx.edge_betweenness_centrality(U, k=K, weight="length", seed=7, normalized=True)
edges["betweenness_s7"] = [eb2.get((u, v, k), eb2.get((v, u, k), 0)) for u, v, k in zip(edges.u, edges.v, edges.key)]
rho = edges[["betweenness", "betweenness_s7"]].corr(method="spearman").iloc[0, 1]
top2 = edges.groupby("nama")["betweenness_s7"].max().sort_values(ascending=False).head(10)
print(f"Spearman antar-seed: {rho:.3f} | jalan yang sama di 10 teratas: {len(set(top.index) & set(top2.index))}/10")

## 3. Closeness
Closeness = kebalikan rerata jarak ke semua simpul lain: "seberapa dekat simpul ini ke seluruh jaringan".
Pada skala kota, closeness hampir selalu memuncak di pusat geometris, jadi orang menghitungnya lokal
(dalam radius tertentu). Hati-hati: closeness dalam radius hanya merata-rata simpul yang *terjangkau*. Di
kawasan jarang, simpul terjangkau sedikit dan dekat — closeness-nya justru tinggi. Lihat tabel di bawah:
Mijen, kecamatan terjarang, mendapat closeness tertinggi. Ukuran lokal yang lebih jujur adalah **reach**:
jumlah simpul yang terjangkau dalam 2 km.

In [ ]:
kec = gpd.read_file(OSM, layer="kecamatan").to_crs(UTM)
sampel = pd.Series(list(U.nodes)).sample(2000, random_state=SEED)
jarak2k = {n: nx.single_source_dijkstra_path_length(U, n, cutoff=2000, weight="length") for n in sampel}
nodes = ox.graph_to_gdfs(U, edges=False).loc[sampel]
nodes["closeness_2km"] = [(len(d) - 1) / sum(d.values()) if len(d) > 1 else 0 for d in jarak2k.values()]
nodes["simpul_2km"] = [len(d) - 1 for d in jarak2k.values()]
nodes = gpd.sjoin(nodes, kec[["kecamatan", "geometry"]], predicate="within")
nodes.groupby("kecamatan")[["simpul_2km", "closeness_2km"]].median() \
    .sort_values("simpul_2km", ascending=False).round({"simpul_2km": 0, "closeness_2km": 5})

## 4. Aksesibilitas jaringan ke puskesmas
Pejalan kaki: jaringan `walk`, 15 menit berjalan (4,5 km/jam ≈ 1.125 m). Bandingkan dua cara:
**buffer garis lurus** 1.125 m dan **jarak sepanjang jalan** 1.125 m.

In [ ]:
W = ox.project_graph(ox.load_graphml(RAW / "jalan_walk.graphml"), to_crs=UTM)
Wu = ox.convert.to_undirected(W)
pk = gpd.read_file(PROC / "puskesmas.gpkg")
# Jarak total = puskesmas → simpul jalan terdekat + sepanjang jaringan + simpul → bangunan.
# Simpul semu "PK" dihubungkan ke simpul terdekat tiap puskesmas dengan bobot jarak sambungannya.
simpul_pk, sambung_pk = ox.distance.nearest_nodes(Wu, pk.geometry.x, pk.geometry.y, return_dist=True)
H = nx.Graph()
H.add_weighted_edges_from(((u, v, d["length"]) for u, v, d in Wu.edges(data=True)), weight="length")
for n, d in zip(simpul_pk, sambung_pk):
    if H.has_edge("PK", n):
        d = min(d, H["PK"][n]["length"])
    H.add_edge("PK", n, length=d)
jarak_pk = nx.single_source_dijkstra_path_length(H, "PK", weight="length")

bangunan = gpd.read_file(PROC / "bangunan_utm.gpkg", layer="bangunan", columns=["id"])
pt = bangunan.representative_point()
bangunan["simpul"], sambung = ox.distance.nearest_nodes(Wu, pt.x, pt.y, return_dist=True)
bangunan["jarak_jaringan"] = bangunan["simpul"].map(jarak_pk) + sambung
print(f"median jarak bangunan → jalan: {np.median(sambung):.0f} m | puskesmas → jalan: {np.median(sambung_pk):.0f} m")
bangunan["jarak_lurus"] = pt.distance(pk.geometry.union_all()).values

R = 1125
pd.Series({
    "dalam buffer 1.125 m (garis lurus)": (bangunan["jarak_lurus"] <= R).mean(),
    "dalam 1.125 m sepanjang jalan": (bangunan["jarak_jaringan"] <= R).mean(),
}, name="bagian bangunan").map("{:.1%}".format)

Selisihnya adalah bangunan yang *tampak* dekat di peta tetapi harus memutar — karena sungai, rel, tol, atau
jalan buntu, **atau** karena gang kampung yang belum dipetakan di OSM (jarak jaringan jadi terlalu besar).
Cek visual di peta sebelum menyimpulkan. Rasio jarak jaringan ÷ jarak lurus menunjukkan di mana hambatan itu.

In [ ]:
dekat = bangunan[bangunan["jarak_lurus"] <= R].copy()
dekat["rasio"] = dekat["jarak_jaringan"] / dekat["jarak_lurus"].clip(lower=50)
dekat = gpd.sjoin(dekat.set_geometry(pt[dekat.index]), kec[["kecamatan", "geometry"]], predicate="within")
dekat.groupby("kecamatan").apply(lambda d: (d["jarak_jaringan"] > R).mean(), include_groups=False) \
    .sort_values(ascending=False).head(6).map("{:.1%}".format).rename("tampak dekat, ternyata > 15 menit")

In [ ]:
edges[["u", "v", "key", "nama", "highway", "length", "betweenness", "betweenness_s7", "geometry"]] \
    .to_file(PROC / "betweenness.gpkg", layer="ruas")
bangunan.drop(columns="simpul").to_file(PROC / "akses_puskesmas.gpkg", layer="bangunan")
kel = gpd.read_file(PROC / "morfologi.gpkg", layer="kelurahan")[["id", "geometry"]].rename(columns={"id": "kel_id"})
bk = gpd.sjoin(bangunan.set_geometry(pt), kel, predicate="within")
bk.groupby("kel_id").agg(lurus_15mnt=("jarak_lurus", lambda s: (s <= R).mean()),
                         jaringan_15mnt=("jarak_jaringan", lambda s: (s <= R).mean())) \
    .reset_index().rename(columns={"kel_id": "id"}).to_csv(PROC / "akses_puskesmas_kelurahan.csv", index=False)
print("disimpan: betweenness.gpkg, akses_puskesmas.gpkg, akses_puskesmas_kelurahan.csv")